# Embedding Tutorial

This tutorial shows how to embed light curves with pretrained ONNX models
from the `light_curve.embed` submodule.

Models are distributed as ONNX files and downloaded from HuggingFace Hub by `from_hf()`.

Requires: `pip install onnxruntime` (and optionally `huggingface_hub` for automatic downloads)

In [ ]:
# %pip install light-curve huggingface_hub onnxruntime

## Astromer2 — single-band embeddings

[Astromer2](https://ui.adsabs.harvard.edu/abs/2026A%26A...707A.170D/abstract) is pretrained on
MACHO light curves and produces 256-dimensional embeddings from irregularly-sampled `(time, mag)` pairs.

In [1]:
import numpy as np
from light_curve.embed import Astromer2

model = Astromer2.from_hf(output="mean")
print(f"Model loaded. Max sequence length: {model.seq_size}")

rng = np.random.default_rng(0)
time = np.sort(rng.uniform(0, 500, 120)).astype(np.float64)
mag  = rng.normal(15, 0.5, 120).astype(np.float64)

embedding = model(time, mag)
print(f"Output shape: {embedding.shape}  # (n_bands, n_subsamples, seq_windows, embed_dim)")
print(f"Squeezed:     {embedding.squeeze().shape}")


Model loaded. Max sequence length: 200
Output shape: (1, 1, 1, 256)  # (n_bands, n_subsamples, seq_windows, embed_dim)
Squeezed:     (256,)


## Astromer2 — multi-band

Pass `bands=[...]` to embed each band independently.
The model returns one embedding per band:

In [2]:
model_gr = Astromer2.from_hf(output="mean", bands=["g", "r"])

rng2 = np.random.default_rng(1)
n = 120
time_gr = np.sort(rng2.uniform(0, 400, n)).astype(np.float64)
mag_gr  = rng2.normal(15, 0.4, n).astype(np.float64)
band_gr = np.array(["g", "r"] * (n // 2))

emb_gr = model_gr(time_gr, mag_gr, band=band_gr)
print(f"Output shape: {emb_gr.shape}  # (2 bands, n_subsamples, seq_windows, embed_dim)")


Output shape: (2, 1, 1, 256)  # (2 bands, n_subsamples, seq_windows, embed_dim)


## ATCAT — 6-band LSST model

[ATCAT](https://ui.adsabs.harvard.edu/abs/2025arXiv251100614T/abstract) processes all ugrizY bands
jointly and returns 384-dimensional embeddings. Inputs are flux, flux error, time, and integer band
index (u=0, g=1, r=2, i=3, z=4, Y=5).

In [3]:
from light_curve.embed import ATCAT

model_atcat = ATCAT.from_hf(output="last")
print(f"ATCAT loaded. Max sequence length: {model_atcat.seq_size}")

rng3 = np.random.default_rng(2)
n3 = 150
time3     = np.sort(rng3.uniform(0, 500, n3)).astype(np.float32)
flux3     = rng3.normal(100, 15, n3).astype(np.float32)  # flux in nJy
flux_err3 = np.full(n3, 5.0, dtype=np.float32)
band3     = np.array([i % 6 for i in range(n3)])  # u=0, g=1, r=2, i=3, z=4, Y=5

emb3 = model_atcat(time3, flux3, flux_err3, band3)
print(f"Output shape: {emb3.shape}  # (1, 1, 1, {emb3.shape[-1]})")


ATCAT loaded. Max sequence length: 243
Output shape: (1, 1, 1, 384)  # (1, 1, 1, 384)


## Notes

- Embeddings have shape `(n_bands, n_subsamples, seq_windows, embed_dim)`. Use `.squeeze()` to get a flat vector for a single object.
- Hardware-accelerated inference is supported through [execution providers](https://onnxruntime.ai/docs/execution-providers/) (CUDA, DirectML, CoreML, OpenVINO, and others). Install the matching `onnxruntime` variant and pass `providers` accordingly — for example, for NVIDIA GPUs:
  ```python
  model = Astromer2.from_hf(
      output="mean",
      ort_session_kwargs={"providers": ["CUDAExecutionProvider"]},
  )
  ```
- `huggingface_hub` is only needed for automatic downloads via `from_hf()`. If you already have the ONNX file, it is not required.
- [API reference](../api/)